# Load Cleaned Datasets

In [2]:
import pandas as pd
import numpy as np

In [3]:
orders = pd.read_csv(
    "../data/cleaned/orders_cleaned.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

order_items = pd.read_csv("../data/cleaned/order_items_cleaned.csv")

customers = pd.read_csv("../data/cleaned/customers_cleaned.csv")

products = pd.read_csv("../data/cleaned/products_cleaned.csv")

payments = pd.read_csv("../data/cleaned/payments_cleaned.csv")

reviews = pd.read_csv(
    "../data/cleaned/reviews_cleaned.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

sellers = pd.read_csv("../data/cleaned/sellers_cleaned.csv")

# Dataset Validation

In [4]:

datasets = {
    "Orders": orders,
    "Order Items": order_items,
    "Customers": customers,
    "Products": products,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers
}

summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1]
    })

pd.DataFrame(summary)

,Dataset,Rows,Columns
0,Orders,99441,8
1,Order Items,112650,7
2,Customers,99441,5
3,Products,32951,9
4,Payments,103886,5
5,Reviews,99224,7
6,Sellers,3095,4


# Order Feature Engineering

In [5]:
order_features = orders.copy()

In [6]:
order_features["purchase_year"] = (
    order_features["order_purchase_timestamp"].dt.year
)

order_features["purchase_month"] = (
    order_features["order_purchase_timestamp"].dt.month
)

order_features["purchase_quarter"] = (
    order_features["order_purchase_timestamp"].dt.quarter
)

order_features["purchase_day"] = (
    order_features["order_purchase_timestamp"].dt.day
)

order_features["purchase_hour"] = (
    order_features["order_purchase_timestamp"].dt.hour
)

order_features["purchase_weekday"] = (
    order_features["order_purchase_timestamp"].dt.day_name()
)

order_features["is_weekend"] = (
    order_features["purchase_weekday"]
    .isin(["Saturday", "Sunday"])
)

In [7]:
order_features[
    [
        "order_purchase_timestamp",
        "purchase_year",
        "purchase_month",
        "purchase_quarter",
        "purchase_day",
        "purchase_hour",
        "purchase_weekday",
        "is_weekend"
    ]
].head()

,order_purchase_timestamp,purchase_year,purchase_month,purchase_quarter,purchase_day,purchase_hour,purchase_weekday,is_weekend
0,2017-10-02 10:56:33,2017,10,4,2,10,Monday,False
1,2018-07-24 20:41:37,2018,7,3,24,20,Tuesday,False
2,2018-08-08 08:38:49,2018,8,3,8,8,Wednesday,False
3,2017-11-18 19:28:06,2017,11,4,18,19,Saturday,True
4,2018-02-13 21:18:39,2018,2,1,13,21,Tuesday,False


In [8]:
order_features["approval_time_hours"] = (
    (
        order_features["order_approved_at"] -
        order_features["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600
).round(2)

In [9]:
order_features["delivery_time_days"] = (
    (
        order_features["order_delivered_customer_date"] -
        order_features["order_purchase_timestamp"]
    ).dt.days
)

In [10]:
order_features["shipping_time_days"] = (
    (
        order_features["order_delivered_customer_date"] -
        order_features["order_delivered_carrier_date"]
    ).dt.days
)

In [11]:
order_features["estimated_delivery_days"] = (
    (
        order_features["order_estimated_delivery_date"] -
        order_features["order_purchase_timestamp"]
    ).dt.days
)

In [12]:
order_features["delivery_delay_days"] = (
    (
        order_features["order_delivered_customer_date"] -
        order_features["order_estimated_delivery_date"]
    ).dt.days
)

In [13]:
order_features["is_delayed"] = (
    order_features["delivery_delay_days"] > 0
)

In [14]:
order_features[
    [
        "delivery_time_days",
        "shipping_time_days",
        "approval_time_hours",
        "estimated_delivery_days",
        "delivery_delay_days",
        "is_delayed"
    ]
].describe()

,delivery_time_days,shipping_time_days,approval_time_hours,estimated_delivery_days,delivery_delay_days
count,96476.000000,96475.000000,99281.000000,99441.000000,96476.000000
mean,12.094086,8.878310,10.419073,23.403958,-11.876881
std,9.551746,8.746088,26.038012,8.829562,10.183854
min,0.000000,-17.000000,0.000000,1.000000,-147.000000
25%,6.000000,4.000000,0.220000,18.000000,-17.000000
50%,10.000000,7.000000,0.340000,23.000000,-12.000000
75%,15.000000,12.000000,14.580000,28.000000,-7.000000
max,209.000000,205.000000,4509.180000,155.000000,188.000000


In [15]:
order_features["approval_time_hours"].describe()

count    99281.000000
mean        10.419073
std         26.038012
min          0.000000
25%          0.220000
50%          0.340000
75%         14.580000
max       4509.180000
Name: approval_time_hours, dtype: float64

In [16]:
order_features["delivery_delay_days"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

In [17]:
from pathlib import Path

order_path = Path("../data/processed/order")
order_path.mkdir(parents=True, exist_ok=True)

order_features.to_csv(
    order_path / "order_features.csv",
    index=False
)

print("Order features exported successfully.")

Order features exported successfully.


In [20]:
validation = {
    "Negative Shipping Time": (order_features["shipping_time_days"] < 0).sum(),
    "Negative Delivery Time": (order_features["delivery_time_days"] < 0).sum(),
    "Negative Approval Time": (order_features["approval_time_hours"] < 0).sum(),
    "Late Deliveries": (order_features["delivery_delay_days"] > 0).sum(),
    "Early Deliveries": (order_features["delivery_delay_days"] < 0).sum()
}

pd.DataFrame(
    validation.items(),
    columns=["Validation Check", "Count"]
)

,Validation Check,Count
0,Negative Shipping Time,23
1,Negative Delivery Time,0
2,Negative Approval Time,0
3,Late Deliveries,6535
4,Early Deliveries,88649


# Build Sales Master Table

In [21]:
sales_master = order_features.copy()

In [22]:
sales_master = sales_master.merge(
    customers,
    on="customer_id",
    how="left"
)

In [23]:
sales_master.shape

(99441, 25)

In [24]:
sales_master = sales_master.merge(
    order_items,
    on="order_id",
    how="left"
)

In [25]:
sales_master.shape

(113425, 31)

In [26]:
sales_master = sales_master.merge(
    products,
    on="product_id",
    how="left"
)

In [27]:
sales_master.shape

(113425, 39)

In [28]:
sales_master = sales_master.merge(
    sellers,
    on="seller_id",
    how="left"
)

In [29]:
sales_master.shape

(113425, 42)

In [30]:
payment_summary = (
    payments
    .groupby("order_id", as_index=False)
    .agg({
        "payment_value": "sum",
        "payment_installments": "max",
        "payment_type": lambda x: x.mode().iloc[0]
    })
)

In [31]:
sales_master = sales_master.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [32]:
review_summary = (
    reviews
    .groupby("order_id", as_index=False)
    .agg({
        "review_score": "mean"
    })
)

In [ ]:
sales_master = sales_master.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [34]:
sales_master.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 46 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   purchase_year                  113425 non-null  int32         
 9   purchase_month                 113425 non-null  int32         
 10  purchase_quarter               113425 non-null  int32         
 11  purchase_da

In [35]:
sales_master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,19.0,8.0,13.0,9350.0,maua,SP,38.71,1.0,voucher,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,19.0,13.0,19.0,31570.0,belo horizonte,SP,141.46,1.0,boleto,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,24.0,19.0,21.0,14840.0,guariba,SP,179.12,3.0,credit_card,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,30.0,10.0,20.0,31842.0,belo horizonte,MG,72.20,1.0,credit_card,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,51.0,15.0,15.0,8752.0,mogi das cruzes,SP,28.62,1.0,credit_card,5.0


In [36]:
sales_master.shape

(113425, 46)

In [37]:
from pathlib import Path

master_path = Path("../data/processed/master")
master_path.mkdir(parents=True, exist_ok=True)

sales_master.to_csv(
    master_path / "sales_master.csv",
    index=False
)

print("✅ Sales Master exported successfully.")

✅ Sales Master exported successfully.


# Customer Feature Engineering

In [39]:
customer_features = customers.copy()
customer_features.shape

(99441, 5)

In [41]:
purchase_summary = (
    sales_master
    .groupby("customer_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_products=("order_item_id", "count"),
        total_spent=("payment_value", "sum"),
        average_order_value=("payment_value", "mean")
    )
)

In [42]:
purchase_summary.head()

,customer_id,total_orders,total_products,total_spent,average_order_value
0,00012a2ce6f8dcda20d059ce98491703,1,1,114.74,114.74
1,000161a058600d5901f007fab4c27140,1,1,67.41,67.41
2,0001fd6190edaaf884bcaf3d49edf079,1,1,195.42,195.42
3,0002414f95344307404f0ace7a26f1d5,1,1,179.35,179.35
4,000379cdec625522490c315e70c7a9fb,1,1,107.01,107.01


In [43]:
customer_features = customer_features.merge(
    purchase_summary,
    on="customer_id",
    how="left"
)

In [44]:
customer_features.shape

(99441, 9)

In [45]:
customer_dates = (
    sales_master
    .groupby("customer_id", as_index=False)
    .agg(
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max")
    )
)

In [46]:
customer_features = customer_features.merge(
    customer_dates,
    on="customer_id",
    how="left"
)

In [47]:
customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase"] -
    customer_features["first_purchase"]
).dt.days

In [48]:
customer_features["purchase_frequency"] = np.where(
    customer_features["customer_lifetime_days"] == 0,
    customer_features["total_orders"],
    (
        customer_features["total_orders"] /
        customer_features["customer_lifetime_days"]
    ).round(4)
)

In [49]:
customer_features["repeat_customer"] = (
    customer_features["total_orders"] > 1
)

In [50]:
review_summary = (
    sales_master
    .groupby("customer_id", as_index=False)
    .agg(
        average_review_score=("review_score", "mean")
    )
)

customer_features = customer_features.merge(
    review_summary,
    on="customer_id",
    how="left"
)

In [52]:
payment_summary = (
    sales_master
    .groupby("customer_id")["payment_type"]
    .agg(
        lambda x: (
            x.dropna().mode().iloc[0]
            if not x.dropna().mode().empty
            else np.nan
        )
    )
    .reset_index(name="preferred_payment_type")
)

customer_features = customer_features.merge(
    payment_summary,
    on="customer_id",
    how="left"
)

In [53]:
customer_features["preferred_payment_type"].value_counts(dropna=False)

preferred_payment_type
credit_card    76132
boleto         19784
voucher         1994
debit_card      1527
not_defined        3
NaN                1
Name: count, dtype: int64

# Build Customer 360 Dataset

In [54]:
customer_360 = (
    sales_master[
        [
            "customer_unique_id",
            "customer_id",
            "customer_city",
            "customer_state"
        ]
    ]
    .drop_duplicates(subset="customer_unique_id")
    .copy()
)

customer_360.shape

(96096, 4)

In [55]:
purchase_summary = (
    sales_master
    .groupby("customer_unique_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_products=("order_item_id", "count"),
        total_spent=("payment_value", "sum"),
        average_order_value=("payment_value", "mean")
    )
)

purchase_summary.head()

,customer_unique_id,total_orders,total_products,total_spent,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,1,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,1,196.89,196.89


In [56]:
customer_360 = customer_360.merge(
    purchase_summary,
    on="customer_unique_id",
    how="left"
)

customer_360.shape

(96096, 8)

In [57]:
customer_dates = (
    sales_master
    .groupby("customer_unique_id", as_index=False)
    .agg(
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max")
    )
)

In [58]:
customer_360 = customer_360.merge(
    customer_dates,
    on="customer_unique_id",
    how="left"
)

In [59]:
customer_360["customer_lifetime_days"] = (
    customer_360["last_purchase"] -
    customer_360["first_purchase"]
).dt.days

In [60]:
customer_360["purchase_frequency"] = np.where(
    customer_360["customer_lifetime_days"] == 0,
    customer_360["total_orders"],
    (
        customer_360["total_orders"] /
        customer_360["customer_lifetime_days"]
    ).round(4)
)

In [61]:
customer_360["repeat_customer"] = (
    customer_360["total_orders"] > 1
)

In [62]:
review_summary = (
    sales_master
    .groupby("customer_unique_id", as_index=False)
    .agg(
        average_review_score=("review_score", "mean")
    )
)

customer_360 = customer_360.merge(
    review_summary,
    on="customer_unique_id",
    how="left"
)

In [63]:
payment_summary = (
    sales_master
    .groupby("customer_unique_id")["payment_type"]
    .agg(
        lambda x: (
            x.dropna().mode().iloc[0]
            if not x.dropna().mode().empty
            else np.nan
        )
    )
    .reset_index(name="preferred_payment_type")
)

customer_360 = customer_360.merge(
    payment_summary,
    on="customer_unique_id",
    how="left"
)

In [64]:
customer_360.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      96096 non-null  str           
 1   customer_id             96096 non-null  str           
 2   customer_city           96096 non-null  str           
 3   customer_state          96096 non-null  str           
 4   total_orders            96096 non-null  int64         
 5   total_products          96096 non-null  int64         
 6   total_spent             96096 non-null  float64       
 7   average_order_value     96095 non-null  float64       
 8   first_purchase          96096 non-null  datetime64[us]
 9   last_purchase           96096 non-null  datetime64[us]
 10  customer_lifetime_days  96096 non-null  int64         
 11  purchase_frequency      96096 non-null  float64       
 12  repeat_customer         96096 non-null  bool          
 1

In [65]:
customer_360.head()

,customer_unique_id,customer_id,customer_city,customer_state,total_orders,total_products,total_spent,average_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency,repeat_customer,average_review_score,preferred_payment_type
0,7c396fd4830fd04220f754e42b4e5bff,9ef432eb6251297304e76186b10a928d,sao paulo,SP,2,2,82.82,41.41,2017-09-04 11:26:38,2017-10-02 10:56:33,27,0.0741,True,4.5,credit_card
1,af07308b275d755c9edb36a90c618231,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,BA,1,1,141.46,141.46,2018-07-24 20:41:37,2018-07-24 20:41:37,0,1.0000,False,4.0,boleto
2,3a653a41f6f9fc3d2a113cf8398680e8,41ce2a54c0b03bf3443c3d931a367089,vianopolis,GO,1,1,179.12,179.12,2018-08-08 08:38:49,2018-08-08 08:38:49,0,1.0000,False,5.0,credit_card
3,7c142cf63193a1473d2e66489a9ae977,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,RN,1,1,72.20,72.20,2017-11-18 19:28:06,2017-11-18 19:28:06,0,1.0000,False,5.0,credit_card
4,72632f0f9dd73dfee390c9b22eb56dd6,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,SP,1,1,28.62,28.62,2018-02-13 21:18:39,2018-02-13 21:18:39,0,1.0000,False,5.0,credit_card


In [66]:
customer_360.describe(include="all")

,customer_unique_id,customer_id,customer_city,customer_state,total_orders,total_products,total_spent,average_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency,repeat_customer,average_review_score,preferred_payment_type
count,96096,96096,96096,96096,96096.000000,96096.000000,96096.000000,96095.000000,96096,96096,96096.000000,96096.000000,96096,95380.000000,96095
unique,96096,96096,4118,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,5
top,7c396fd4830fd04220f754e42b4e5bff,9ef432eb6251297304e76186b10a928d,sao paulo,SP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,credit_card
freq,1,1,14974,40296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93099,NaN,73553
mean,NaN,NaN,NaN,NaN,1.034809,1.172265,213.023712,161.539532,2017-12-30 19:19:10.429206,2018-01-02 12:40:19.655864,2.711507,0.990012,NaN,4.084671,NaN
min,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,0.000000,2016-09-04 21:15:19,2016-09-04 21:15:19,0.000000,0.003300,NaN,1.000000,NaN
25%,NaN,NaN,NaN,NaN,1.000000,1.000000,63.990000,62.480000,2017-09-11 19:52:06,2017-09-15 09:04:17.250000,0.000000,1.000000,NaN,4.000000,NaN
50%,NaN,NaN,NaN,NaN,1.000000,1.000000,113.150000,105.860000,2018-01-18 13:33:08,2018-01-21 19:39:16,0.000000,1.000000,NaN,5.000000,NaN
75%,NaN,NaN,NaN,NaN,1.000000,1.000000,202.732500,177.360000,2018-05-04 10:38:45,2018-05-06 20:14:49.750000,0.000000,1.000000,NaN,5.000000,NaN
max,NaN,NaN,NaN,NaN,17.000000,24.000000,109312.640000,13664.080000,2018-10-17 17:30:18,2018-10-17 17:30:18,633.000000,6.000000,NaN,5.000000,NaN


In [67]:
customer_360["customer_unique_id"].is_unique

True

In [68]:
customer_360 = customer_360[
    [
        "customer_unique_id",
        "customer_id",
        "customer_city",
        "customer_state",

        "first_purchase",
        "last_purchase",
        "customer_lifetime_days",

        "total_orders",
        "total_products",
        "total_spent",
        "average_order_value",
        "purchase_frequency",

        "repeat_customer",
        "average_review_score",
        "preferred_payment_type"
    ]
]

customer_360.head()

,customer_unique_id,customer_id,customer_city,customer_state,first_purchase,last_purchase,customer_lifetime_days,total_orders,total_products,total_spent,average_order_value,purchase_frequency,repeat_customer,average_review_score,preferred_payment_type
0,7c396fd4830fd04220f754e42b4e5bff,9ef432eb6251297304e76186b10a928d,sao paulo,SP,2017-09-04 11:26:38,2017-10-02 10:56:33,27,2,2,82.82,41.41,0.0741,True,4.5,credit_card
1,af07308b275d755c9edb36a90c618231,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,BA,2018-07-24 20:41:37,2018-07-24 20:41:37,0,1,1,141.46,141.46,1.0000,False,4.0,boleto
2,3a653a41f6f9fc3d2a113cf8398680e8,41ce2a54c0b03bf3443c3d931a367089,vianopolis,GO,2018-08-08 08:38:49,2018-08-08 08:38:49,0,1,1,179.12,179.12,1.0000,False,5.0,credit_card
3,7c142cf63193a1473d2e66489a9ae977,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,RN,2017-11-18 19:28:06,2017-11-18 19:28:06,0,1,1,72.20,72.20,1.0000,False,5.0,credit_card
4,72632f0f9dd73dfee390c9b22eb56dd6,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,SP,2018-02-13 21:18:39,2018-02-13 21:18:39,0,1,1,28.62,28.62,1.0000,False,5.0,credit_card


In [101]:
from pathlib import Path

customer_path = Path("../data/processed/customer")
customer_path.mkdir(parents=True, exist_ok=True)

customer_360.to_csv(
    customer_path / "customer_360.csv",
    index=False
)

print("✅ customer_360.csv exported successfully.")

✅ customer_360.csv exported successfully.


# Build Customer Features Dataset

In [69]:
customer_features = customer_360.copy()

customer_features.shape

(96096, 15)

In [70]:
reference_date = (
    customer_features["last_purchase"].max() +
    pd.Timedelta(days=1)
)

customer_features["recency_days"] = (
    reference_date -
    customer_features["last_purchase"]
).dt.days

In [71]:
customer_features["recency_days"].describe()

count    96096.000000
mean       288.735691
std        153.414676
min          1.000000
25%        164.000000
50%        269.000000
75%        398.000000
max        773.000000
Name: recency_days, dtype: float64

In [72]:
customer_features["monetary_segment"] = pd.qcut(
    customer_features["total_spent"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

In [73]:
customer_features["monetary_segment"].value_counts()

monetary_segment
Low        24026
Medium     24025
Premium    24024
High       24021
Name: count, dtype: int64

In [78]:
customer_features["total_orders"].value_counts().head(10)

total_orders
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
17        1
9         1
Name: count, dtype: int64

In [79]:
customer_features["frequency_segment"] = np.select(
    [
        customer_features["total_orders"] == 1,
        customer_features["total_orders"].between(2, 3),
        customer_features["total_orders"].between(4, 6),
        customer_features["total_orders"] >= 7
    ],
    [
        "One-Time",
        "Occasional",
        "Frequent",
        "Loyal"
    ],
    default="Unknown"
)

In [80]:
customer_features["frequency_segment"].value_counts()

frequency_segment
One-Time      93099
Occasional     2948
Frequent         44
Loyal             5
Name: count, dtype: int64

In [81]:
q1 = customer_features["total_spent"].quantile(0.25)
q2 = customer_features["total_spent"].quantile(0.50)
q3 = customer_features["total_spent"].quantile(0.75)

customer_features["customer_value_tier"] = np.select(
    [
        customer_features["total_spent"] <= q1,
        customer_features["total_spent"] <= q2,
        customer_features["total_spent"] <= q3,
        customer_features["total_spent"] > q3
    ],
    [
        "Budget",
        "Regular",
        "Premium",
        "VIP"
    ],
    default="Unknown"
)

In [82]:
customer_features["customer_value_tier"].value_counts()

customer_value_tier
Budget     24026
Regular    24025
VIP        24024
Premium    24021
Name: count, dtype: int64

In [83]:
customer_features["review_category"] = np.select(
    [
        customer_features["average_review_score"] <= 2,
        customer_features["average_review_score"] == 3,
        customer_features["average_review_score"] == 4,
        customer_features["average_review_score"] >= 5
    ],
    [
        "Poor",
        "Average",
        "Good",
        "Excellent"
    ],
    default="Not Rated"
)

In [84]:
customer_features["review_category"].value_counts()

review_category
Excellent    54604
Good         18300
Poor         13869
Average       7865
Not Rated     1458
Name: count, dtype: int64

In [85]:
customer_features["normalized_orders"] = (
    customer_features["total_orders"] /
    customer_features["total_orders"].max()
)

customer_features["normalized_spent"] = (
    customer_features["total_spent"] /
    customer_features["total_spent"].max()
)

customer_features["loyalty_score"] = (
    (
        customer_features["normalized_orders"] * 0.5 +
        customer_features["normalized_spent"] * 0.5
    ) * 100
).round(2)

In [93]:
customer_features["loyalty_segment"] = np.select(
    [
        customer_features["loyalty_score"] < 25,
        customer_features["loyalty_score"] < 50,
        customer_features["loyalty_score"] < 75,
        customer_features["loyalty_score"] >= 75
    ],
    [
        "Bronze",
        "Silver",
        "Gold",
        "Platinum"
    ],
    default="Unknown"
)

In [94]:
customer_features[
    [
        "loyalty_score",
        "loyalty_segment"
    ]
].head()

,loyalty_score,loyalty_segment
0,5.92,Bronze
1,3.01,Bronze
2,3.02,Bronze
3,2.97,Bronze
4,2.95,Bronze


In [95]:
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      96096 non-null  str           
 1   customer_id             96096 non-null  str           
 2   customer_city           96096 non-null  str           
 3   customer_state          96096 non-null  str           
 4   first_purchase          96096 non-null  datetime64[us]
 5   last_purchase           96096 non-null  datetime64[us]
 6   customer_lifetime_days  96096 non-null  int64         
 7   total_orders            96096 non-null  int64         
 8   total_products          96096 non-null  int64         
 9   total_spent             96096 non-null  float64       
 10  average_order_value     96095 non-null  float64       
 11  purchase_frequency      96096 non-null  float64       
 12  repeat_customer         96096 non-null  bool          
 1

In [96]:
customer_features.describe(include="all")

,customer_unique_id,customer_id,customer_city,customer_state,first_purchase,last_purchase,customer_lifetime_days,total_orders,total_products,total_spent,...,preferred_payment_type,recency_days,monetary_segment,frequency_segment,customer_value_tier,review_category,normalized_orders,normalized_spent,loyalty_score,loyalty_segment
count,96096,96096,96096,96096,96096,96096,96096.000000,96096.000000,96096.000000,96096.000000,...,96095,96096.000000,96096,96096,96096,96096,96096.000000,96096.000000,96096.000000,96096
unique,96096,96096,4118,27,NaN,NaN,NaN,NaN,NaN,NaN,...,5,NaN,4,4,4,5,NaN,NaN,NaN,3
top,7c396fd4830fd04220f754e42b4e5bff,9ef432eb6251297304e76186b10a928d,sao paulo,SP,NaN,NaN,NaN,NaN,NaN,NaN,...,credit_card,NaN,Low,One-Time,Budget,Excellent,NaN,NaN,NaN,Bronze
freq,1,1,14974,40296,NaN,NaN,NaN,NaN,NaN,NaN,...,73553,NaN,24026,93099,24026,54604,NaN,NaN,NaN,96093
mean,NaN,NaN,NaN,NaN,2017-12-30 19:19:10.429206,2018-01-02 12:40:19.655864,2.711507,1.034809,1.172265,213.023712,...,NaN,288.735691,NaN,NaN,NaN,NaN,0.060871,0.001949,3.141001,NaN
min,NaN,NaN,NaN,NaN,2016-09-04 21:15:19,2016-09-04 21:15:19,0.000000,1.000000,0.000000,0.000000,...,NaN,1.000000,NaN,NaN,NaN,NaN,0.058824,0.000000,2.940000,NaN
25%,NaN,NaN,NaN,NaN,2017-09-11 19:52:06,2017-09-15 09:04:17.250000,0.000000,1.000000,1.000000,63.990000,...,NaN,164.000000,NaN,NaN,NaN,NaN,0.058824,0.000585,2.970000,NaN
50%,NaN,NaN,NaN,NaN,2018-01-18 13:33:08,2018-01-21 19:39:16,0.000000,1.000000,1.000000,113.150000,...,NaN,269.000000,NaN,NaN,NaN,NaN,0.058824,0.001035,2.990000,NaN
75%,NaN,NaN,NaN,NaN,2018-05-04 10:38:45,2018-05-06 20:14:49.750000,0.000000,1.000000,1.000000,202.732500,...,NaN,398.000000,NaN,NaN,NaN,NaN,0.058824,0.001855,3.040000,NaN
max,NaN,NaN,NaN,NaN,2018-10-17 17:30:18,2018-10-17 17:30:18,633.000000,17.000000,24.000000,109312.640000,...,NaN,773.000000,NaN,NaN,NaN,NaN,1.000000,1.000000,52.940000,NaN


In [97]:
customer_features.head()

,customer_unique_id,customer_id,customer_city,customer_state,first_purchase,last_purchase,customer_lifetime_days,total_orders,total_products,total_spent,...,preferred_payment_type,recency_days,monetary_segment,frequency_segment,customer_value_tier,review_category,normalized_orders,normalized_spent,loyalty_score,loyalty_segment
0,7c396fd4830fd04220f754e42b4e5bff,9ef432eb6251297304e76186b10a928d,sao paulo,SP,2017-09-04 11:26:38,2017-10-02 10:56:33,27,2,2,82.82,...,credit_card,381,Medium,Occasional,Regular,Not Rated,0.117647,0.000758,5.92,Bronze
1,af07308b275d755c9edb36a90c618231,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,BA,2018-07-24 20:41:37,2018-07-24 20:41:37,0,1,1,141.46,...,boleto,85,High,One-Time,Premium,Good,0.058824,0.001294,3.01,Bronze
2,3a653a41f6f9fc3d2a113cf8398680e8,41ce2a54c0b03bf3443c3d931a367089,vianopolis,GO,2018-08-08 08:38:49,2018-08-08 08:38:49,0,1,1,179.12,...,credit_card,71,High,One-Time,Premium,Excellent,0.058824,0.001639,3.02,Bronze
3,7c142cf63193a1473d2e66489a9ae977,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,RN,2017-11-18 19:28:06,2017-11-18 19:28:06,0,1,1,72.20,...,credit_card,333,Medium,One-Time,Regular,Excellent,0.058824,0.000660,2.97,Bronze
4,72632f0f9dd73dfee390c9b22eb56dd6,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,SP,2018-02-13 21:18:39,2018-02-13 21:18:39,0,1,1,28.62,...,credit_card,246,Low,One-Time,Budget,Excellent,0.058824,0.000262,2.95,Bronze


In [98]:
print("Rows :", len(customer_features))
print("Unique Customers :", customer_features["customer_unique_id"].nunique())
print("Duplicate Customers :", customer_features.duplicated("customer_unique_id").sum())

Rows : 96096
Unique Customers : 96096


Duplicate Customers : 0


In [100]:
from pathlib import Path

customer_path = Path("../data/processed/customer")
customer_path.mkdir(parents=True, exist_ok=True)

customer_features.to_csv(
    customer_path / "customer_features.csv",
    index=False
)

print(" customer_features.csv exported successfully.")


 customer_features.csv exported successfully.


# Product Feature Engineering

In [102]:
product_features = products.copy()

product_features.shape

(32951, 9)

In [103]:
product_summary = (
    sales_master
    .groupby("product_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_quantity_sold=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        average_product_price=("price", "mean"),
        average_freight=("freight_value", "mean"),
        average_review_score=("review_score", "mean")
    )
)

product_summary.head()

,product_id,total_orders,total_quantity_sold,total_revenue,average_product_price,average_freight,average_review_score
0,00066f42aeeb9f3007548bb9d3f33c38,1,1,101.65,101.65,18.59,5.0
1,00088930e925c41fd95ebfe695fd2655,1,1,129.90,129.90,13.93,4.0
2,0009406fd7479715e4bef61dd91f2462,1,1,229.00,229.00,13.10,1.0
3,000b8f95fcb9e0096488278317764d19,2,2,117.80,58.90,19.60,5.0
4,000d9be29b5207b54e86aa1b1ac54872,1,1,199.00,199.00,19.27,5.0


In [104]:
product_features = product_features.merge(
    product_summary,
    on="product_id",
    how="left"
)

In [105]:
product_features.shape

(32951, 15)

In [106]:
q1 = product_features["total_revenue"].quantile(0.25)
q2 = product_features["total_revenue"].quantile(0.50)
q3 = product_features["total_revenue"].quantile(0.75)

product_features["revenue_category"] = np.select(
    [
        product_features["total_revenue"] <= q1,
        product_features["total_revenue"] <= q2,
        product_features["total_revenue"] <= q3,
        product_features["total_revenue"] > q3
    ],
    [
        "Low",
        "Medium",
        "High",
        "Top Performer"
    ],
    default="Unknown"
)

In [107]:
product_features["rating_category"] = np.select(
    [
        product_features["average_review_score"] <= 2,
        product_features["average_review_score"] == 3,
        product_features["average_review_score"] == 4,
        product_features["average_review_score"] >= 5
    ],
    [
        "Poor",
        "Average",
        "Good",
        "Excellent"
    ],
    default="Not Rated"
)

In [108]:
product_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
 9   total_orders                32951 non-null  int64  
 10  total_quantity_sold         32951 non-null  int64  
 11  total_revenue               32951 non-null  float64
 12  average_product_price       32951 non-null  float64
 13  average_freight             32951 non-null

In [109]:
product_features.describe(include="all")

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,total_orders,total_quantity_sold,total_revenue,average_product_price,average_freight,average_review_score,revenue_category,rating_category
count,32951,32341,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000,32951.000000,32951.000000,32951.000000,32951.000000,32951.000000,32789.000000,32951,32951
unique,32951,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,5
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low,Excellent
freq,1,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8333,14009
mean,NaN,NaN,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728,3.108403,3.418713,412.480462,145.302464,21.202166,4.047867,NaN,NaN
std,NaN,NaN,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047,9.456937,10.619709,1371.945598,246.895756,18.089833,1.213893,NaN,NaN
min,NaN,NaN,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,1.000000,1.000000,2.200000,0.850000,0.010000,1.000000,NaN,NaN
25%,NaN,NaN,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000,1.000000,1.000000,59.900000,39.900000,13.600000,3.600000,NaN,NaN
50%,NaN,NaN,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000,1.000000,1.000000,136.750000,79.000000,16.705000,4.500000,NaN,NaN
75%,NaN,NaN,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000,2.000000,3.000000,329.000000,154.900000,21.828333,5.000000,NaN,NaN


In [110]:
print("Rows:", len(product_features))
print("Unique Products:", product_features["product_id"].nunique())
print("Duplicate Products:", product_features.duplicated("product_id").sum())

Rows: 32951
Unique Products: 32951
Duplicate Products: 0


In [111]:
from pathlib import Path

product_path = Path("../data/processed/product")
product_path.mkdir(parents=True, exist_ok=True)

product_features.to_csv(
    product_path / "product_features.csv",
    index=False
)

print("✅ product_features.csv exported successfully.")

✅ product_features.csv exported successfully.


# Seller Feature Engineering

In [112]:
seller_features = sellers.copy()

seller_features.shape

(3095, 4)

In [113]:
seller_summary = (
    sales_master
    .groupby("seller_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_products_sold=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        average_product_price=("price", "mean"),
        average_freight=("freight_value", "mean"),
        average_review_score=("review_score", "mean"),
        average_delivery_time=("delivery_time_days", "mean")
    )
)

seller_summary.head()

,seller_id,total_orders,total_products_sold,total_revenue,average_product_price,average_freight,average_review_score,average_delivery_time
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,895.000000,21.020000,3.666667,10.333333
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,104.937364,37.046611,3.902542,12.628205
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,250.000000,17.940000,1.000000,NaN
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,22.445455,14.430182,3.981818,15.777778
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,120.000000,19.380000,5.000000,4.000000


In [114]:
seller_features = seller_features.merge(
    seller_summary,
    on="seller_id",
    how="left"
)

In [115]:
q1 = seller_features["total_revenue"].quantile(0.25)
q2 = seller_features["total_revenue"].quantile(0.50)
q3 = seller_features["total_revenue"].quantile(0.75)

seller_features["seller_tier"] = np.select(
    [
        seller_features["total_revenue"] <= q1,
        seller_features["total_revenue"] <= q2,
        seller_features["total_revenue"] <= q3,
        seller_features["total_revenue"] > q3
    ],
    [
        "Emerging",
        "Growing",
        "Established",
        "Top Seller"
    ],
    default="Unknown"
)

In [116]:
seller_features["seller_rating"] = np.select(
    [
        seller_features["average_review_score"] <= 2,
        seller_features["average_review_score"] == 3,
        seller_features["average_review_score"] == 4,
        seller_features["average_review_score"] >= 5
    ],
    [
        "Poor",
        "Average",
        "Good",
        "Excellent"
    ],
    default="Not Rated"
)

In [117]:
seller_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   seller_id               3095 non-null   str    
 1   seller_zip_code_prefix  3095 non-null   int64  
 2   seller_city             3095 non-null   str    
 3   seller_state            3095 non-null   str    
 4   total_orders            3095 non-null   int64  
 5   total_products_sold     3095 non-null   int64  
 6   total_revenue           3095 non-null   float64
 7   average_product_price   3095 non-null   float64
 8   average_freight         3095 non-null   float64
 9   average_review_score    3090 non-null   float64
 10  average_delivery_time   2970 non-null   float64
 11  seller_tier             3095 non-null   str    
 12  seller_rating           3095 non-null   str    
dtypes: float64(5), int64(3), str(5)
memory usage: 500.1 KB


In [119]:
seller_features.describe(include="all")

,seller_id,seller_zip_code_prefix,seller_city,seller_state,total_orders,total_products_sold,total_revenue,average_product_price,average_freight,average_review_score,average_delivery_time,seller_tier,seller_rating
count,3095,3095.000000,3095,3095,3095.000000,3095.000000,3095.000000,3095.000000,3095.000000,3090.000000,2970.000000,3095,3095
unique,3095,NaN,611,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,5
top,3442f8959a84dea7ee197c632cb2df15,NaN,sao paulo,SP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Growing,Not Rated
freq,1,NaN,694,1849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,774,2057
mean,NaN,32291.059451,NaN,NaN,32.313409,36.397415,4391.484233,176.325102,23.380116,3.972987,11.705458,NaN,NaN
std,NaN,32713.453830,NaN,NaN,105.139763,119.193461,13921.997192,322.145495,18.957766,0.971256,7.111951,NaN,NaN
min,NaN,1001.000000,NaN,NaN,1.000000,1.000000,3.500000,3.500000,1.200000,1.000000,1.000000,NaN,NaN
25%,NaN,7093.500000,NaN,NaN,2.000000,2.000000,208.850000,52.183608,14.740000,3.714286,7.944504,NaN,NaN
50%,NaN,14940.000000,NaN,NaN,6.000000,8.000000,821.480000,95.465385,18.230000,4.166667,10.691851,NaN,NaN
75%,NaN,64552.500000,NaN,NaN,21.500000,24.000000,3280.830000,173.992857,24.368333,4.600000,13.828125,NaN,NaN


In [118]:
print("Rows:", len(seller_features))
print("Unique Sellers:", seller_features["seller_id"].nunique())
print("Duplicate Sellers:", seller_features.duplicated("seller_id").sum())

Rows: 3095
Unique Sellers: 3095
Duplicate Sellers: 0


In [120]:
from pathlib import Path

seller_path = Path("../data/processed/seller")
seller_path.mkdir(parents=True, exist_ok=True)

seller_features.to_csv(
    seller_path / "seller_features.csv",
    index=False
)

print("✅ seller_features.csv exported successfully.")

✅ seller_features.csv exported successfully.


In [121]:
# Phase 5 Validation

In [122]:
processed = {
    "Order Features": order_features,
    "Sales Master": sales_master,
    "Customer 360": customer_360,
    "Customer Features": customer_features,
    "Product Features": product_features,
    "Seller Features": seller_features
}

summary = []

for name, df in processed.items():
    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Values": int(df.isna().sum().sum())
    })

validation_summary = pd.DataFrame(summary)

validation_summary

,Dataset,Rows,Columns,Duplicate Rows,Missing Values
0,Order Features,99441,21,0,13964
1,Sales Master,113425,46,0,35836
2,Customer 360,96096,15,0,718
3,Customer Features,96096,24,0,718
4,Product Features,32951,17,0,2610
5,Seller Features,3095,13,0,130


In [123]:
validation_summary.to_csv(
    "../reports/feature_engineering_validation.csv",
    index=False
)

print("✅ Validation summary exported successfully.")

✅ Validation summary exported successfully.
